In [2]:
# run_my_scenarios_with_prof_solver.py
# 目的：
# 1) 让你在这里 DIY 小规模场景（无需 CSV）；
# 2) 自动把场景转成导师代码所需的全局：num_scenarios / df / sp / plot_dir；
# 3) 复用导师的 build_pid_model + GurobiLBLowerBounder + snoglode 求解；
# 4) 用包装器把每个场景的概率设定为你在 scenarios 里给的 "prob"。

import os
import numpy as np
import pandas as pd
import pyomo.environ as pyo

# ========= 修改这里：填入你导师那份包含 build_pid_model(...) 的模块名 =========
# 例如：导师文件叫 stochastic_pid_prof.py，则 PROF_MODULE = "stochastic_pid_prof"
PROF_MODULE = "aaa"   # <<< 改成实际模块名

# --------------- 导入导师代码（build_pid_model / 下界类 / snoglode） ---------------
prof = __import__(PROF_MODULE, fromlist=["*"])
# 需要的对象：build_pid_model, GurobiLBLowerBounder, sno, get_solver/ipopt 工具等
build_pid_model_prof = prof.build_pid_model
GurobiLBLowerBounder = prof.GurobiLBLowerBounder
sno = prof.sno
get_solver = prof.get_solver
ipopt = get_solver("ipopt")

# ============= 你的可配置部分（DIY 场景、时间轴等） =============
# 统一的时间轴设置：总时长与离散段数；导师示例用 T=15, nfe=20（步长 0.75）
T_HORIZON = 15.0
NFE = 20
H = T_HORIZON / NFE
times = [i * H for i in range(NFE + 1)]  # 0, H, 2H, ..., T_HORIZON

# 自定义扰动与设定值（示例函数；你也可直接给 list）
def step_sp(t, step_time=3.0, low=0.0, high=0.5):
    return high if t >= step_time else low

def make_d(level=0.2, start=5.0, end=10.0):
    # 在 [start, end] 区间施加常量扰动 level
    return [level if (t >= start and t <= end) else 0.0 for t in times]

# -------------- 在这里 DIY 你的 scenarios（与示例一致） --------------
scenarios = {
    1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
    2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
    3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
}
# ===============================================================

# ============= 适配层：把 scenarios → 导师代码的全局 =============
# 导师 build_pid_model(scenario_name) 期望：
# - 全局 num_scenarios, df（含 tau_xs, tau_us, tau_ds, disturbance_0..NFE）, sp（常数）
# - scenario_name 形如 "scen_i"，它会用此名字去 df.iloc[...] 取行
#
# 注意：导师示例里 setpoint_change 最终被写死为全局 sp（常数）。
# 若你给了时变 sp[t]，此处暂取 sp_const = sp_seq[0] 用于所有场景（与导师示例对齐）。
# 如需时变设定值，需要改导师函数本体，将 x_setpoint 改为 Param(m.t)。

def _resample_to_grid(values, dst_len):
    """把任意长度序列线性重采样到目标长度 dst_len（一般为 NFE+1）。"""
    values = list(values)
    if len(values) == dst_len:
        return [float(v) for v in values]
    x_src = np.linspace(0.0, 1.0, len(values))
    x_dst = np.linspace(0.0, 1.0, dst_len)
    return list(np.interp(x_dst, x_src, values))

def prepare_prof_globals_from_scenarios(scenarios_dict, nfe=NFE, times_list=times):
    rows = []
    scen_names = []

    # 取全局常数 setpoint（与导师代码一致：他们用常数 sp）
    first_key = next(iter(scenarios_dict.keys()))
    first_sp = scenarios_dict[first_key]["sp"]
    sp_const = first_sp[0] if isinstance(first_sp, (list, tuple, np.ndarray)) else float(first_sp)

    # 0 基 scen_i 命名
    for idx, key in enumerate(sorted(scenarios_dict.keys())):
        s = scenarios_dict[key]
        Ku, tau = float(s["Ku"]), float(s["tau"])
        d_seq = _resample_to_grid(s["d"], nfe + 1)

        # 三系数映射：x' = -tau_xs x + tau_us u + tau_ds d
        tau_xs = 1.0 / tau
        tau_us = Ku / tau
        tau_ds = 1.0 / tau

        row = {
            "tau_xs": tau_xs,
            "tau_us": tau_us,
            "tau_ds": tau_ds,
            "scenario_name": f"scen_{idx}",
            "setpoint_change": float(sp_const),  # 虽然导师函数里没直接用这列，但留着对齐
            "probability": float(s.get("prob", 1.0 / len(scenarios_dict))),
        }
        for i in range(nfe + 1):
            row[f"disturbance_{i}"] = float(d_seq[i])

        rows.append(row)
        scen_names.append(row["scenario_name"])

    # 写入导师模块的全局变量（他们的 build_pid_model 会读取这些）
    prof.num_scenarios = len(rows)
    prof.df = pd.DataFrame(rows)
    prof.sp = float(sp_const)  # 关键：导师代码内部把 setpoint 当成全局常数 sp
    prof.plot_dir = os.getcwd() + "/plots_snoglode_parallel/"
    os.makedirs(prof.plot_dir, exist_ok=True)

    return scen_names


# ============= 概率包装器：把导师默认概率替换为 scenarios 里给的 prob =============
def build_pid_model_with_prob(scenario_name):
    m, first_stage, _prob_default = build_pid_model_prof(scenario_name)
    _, scen_num_str = scenario_name.split("_")
    scen_idx = int(scen_num_str)          # 0 基
    row = prof.df.iloc[scen_idx]          # <<< 不再减 1
    my_prob = float(row["probability"])
    return [m, first_stage, my_prob]


# ============= 主流程：准备 → 配置求解器 → 开始求解 =============
def main():
    # 1) 把你的 scenarios 转成导师代码用的全局
    scen_names = prepare_prof_globals_from_scenarios(scenarios, nfe=NFE, times_list=times)

    # 2) 创建 snoglode 参数（与导师脚本一致，只改 subproblem_creator）
    nonconvex_gurobi = pyo.SolverFactory("gurobi")
    nonconvex_gurobi.options["NonConvex"] = 2

    nonconvex_gurobi_lb = pyo.SolverFactory("gurobi")
    nonconvex_gurobi_lb.options["NonConvex"] = 2
    nonconvex_gurobi_lb.options["MIPGap"] = 1e-2
    nonconvex_gurobi_lb.options["TimeLimit"] = 30

    obbt_solver_opts = {
        "NonConvex": 2,
        "MIPGap": 1,
        "TimeLimit": 5
    }

    # 用我们的包装器替代导师原来的 build_pid_model，使其返回自定义概率
    params = sno.SolverParameters(
        subproblem_names = scen_names,
        subproblem_creator = build_pid_model_with_prob,  # <<<< 关键：换成包装器
        lb_solver = nonconvex_gurobi_lb,
        cg_solver = ipopt,
        ub_solver = nonconvex_gurobi
    )
    params.set_bounders(candidate_solution_finder = sno.SolveExtensiveForm,
                        lower_bounder = GurobiLBLowerBounder)
    params.set_bounds_tightening(fbbt=True, obbt=True, obbt_solver_opt=obbt_solver_opts)
    params.set_branching(selection_strategy = sno.HybridBranching,
                         partition_strategy = sno.ExpectedValue)
    params.activate_verbose()

    # 3) 求解
    solver = sno.Solver(params)
    solver.solve(max_iter=1000, rel_tolerance=1e-3, time_limit=600*6)

    # 4)（可选）打印一阶段解并画图（直接复用导师脚本中那段）
    #    这里略；如需完全同款输出，可把导师文件末尾那段拷贝过来使用 solver.solution。

if __name__ == "__main__":
    main()

Generating the models for the subproblems.
	Scenario scen_0 has 3 first stage vars.
	Scenario scen_1 has 3 first stage vars.
	Scenario scen_2 has 3 first stage vars.
Finished generating subproblem models.
Node rooted.
Using EF method for candidate solution. No EF bound. Building EF.
EF built.
here is full solution
scen_0: {'scen_0.K_p': 5.843121453609823, 'scen_0.K_i': 94.65983224234995, 'scen_0.K_d': 984.3299035451358, 'scen_0.x_s[0]': 0.0, 'scen_0.x_s[0.75]': 0.0, 'scen_0.x_s[1.5]': 0.0, 'scen_0.x_s[2.25]': 0.0, 'scen_0.x_s[3.0]': 0.0, 'scen_0.x_s[3.75]': 0.0, 'scen_0.x_s[4.5]': 0.0, 'scen_0.x_s[5.25]': -4.802881420218341e-05, 'scen_0.x_s[6.0]': -9.095392246916134e-05, 'scen_0.x_s[6.75]': -0.00012686113254867237, 'scen_0.x_s[7.5]': -0.00015430070001665186, 'scen_0.x_s[8.25]': -0.00017233587964504693, 'scen_0.x_s[9.0]': -0.0001805625624236029, 'scen_0.x_s[9.75]': -0.00017910058751846591, 'scen_0.x_s[10.5]': -0.00012052992050559652, 'scen_0.x_s[11.25]': -5.9022716001333464e-05, 'scen_0

In [ ]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.opt import SolverFactory
from pyomo.core.base import TransformationFactory
from pyomo.opt import SolverStatus, TerminationCondition
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Tuple, Optional
import numpy as np
import math
import matplotlib.pyplot as plt
import bisect
import itertools as it
from tqdm import tqdm
import csv
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from typing import List, Tuple
import pandas as pd
import time
from pyomo.core.base import TransformationFactory
from pyomo.contrib.piecewise import PiecewiseLinearFunction as PLF
from scipy.interpolate import griddata
from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import plotly.express as px

In [8]:
def build_model_and_first_stage_lists(scenarios_dict):
    scen_names = prepare_prof_globals_from_scenarios(scenarios_dict, nfe=NFE, times_list=times)

    model_list = []
    first_stg_vars_list = []
    for scen_name in scen_names:
        m, first_stage, prob = build_pid_model_with_prob(scen_name)

        # 转换：如果 first_stage 是字符串，就从模型里取组件
        if isinstance(first_stage[0], str):
            resolved = [m.find_component(n) for n in first_stage]
        else:
            resolved = first_stage

        model_list.append(m)
        first_stg_vars_list.append(resolved)

    return model_list, first_stg_vars_list
model_list, first_stg_vars_list = build_model_and_first_stage_lists(scenarios)

KeyError: 0

In [7]:
scenarios = {
    1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
    2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
    3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
}

def build_model_and_first_stage_lists(scenarios_dict):
    """
    输入: scenarios 字典
    输出: (model_list, first_stg_vars_list)
          - model_list: [m1, m2, m3, ...]
          - first_stg_vars_list: [[Kp1, Ki1, Kd1], [Kp2, Ki2, Kd2], ...]
    """
    # 1) 先把 scenarios 转成导师代码用的全局
    scen_names = prepare_prof_globals_from_scenarios(scenarios_dict, nfe=NFE, times_list=times)

    # 2) 遍历每个场景，调用包装好的 build_pid_model_with_prob
    model_list = []
    first_stg_vars_list = []
    for scen_name in scen_names:
        m, first_stage, prob = build_pid_model_with_prob(scen_name)
        model_list.append(m)
        first_stg_vars_list.append(first_stage)

    return model_list, first_stg_vars_list


model_list, first_stg_vars_list = build_model_and_first_stage_lists(scenarios)

print(model_list)
print(first_stg_vars_list)

# calculate nodes with their values
n_per_axis = 3
first_stg_nodes = build_full_grid_points(first_stg_vars_list[0], n_per_axis=n_per_axis, endpoint=True, round_ndigits=12)
node_vals = []
for pt in first_stg_nodes:
    val = 0
    for i in range(N):
        val += evaluate_Q_at(model_list[i], first_stg_vars_list[i], pt, solver)



[<pyomo.core.base.PyomoModel.ConcreteModel object at 0x000001C39730AD50>, <pyomo.core.base.PyomoModel.ConcreteModel object at 0x000001C39A2506E0>, <pyomo.core.base.PyomoModel.ConcreteModel object at 0x000001C39A2CADA0>]
[{'K_p': <pyomo.core.base.var.ScalarVar object at 0x000001C399BB8CD0>, 'K_i': <pyomo.core.base.var.ScalarVar object at 0x000001C3997C2FD0>, 'K_d': <pyomo.core.base.var.ScalarVar object at 0x000001C399BB81D0>}, {'K_p': <pyomo.core.base.var.ScalarVar object at 0x000001C39A0568D0>, 'K_i': <pyomo.core.base.var.ScalarVar object at 0x000001C399CC8ED0>, 'K_d': <pyomo.core.base.var.ScalarVar object at 0x000001C399CC5950>}, {'K_p': <pyomo.core.base.var.ScalarVar object at 0x000001C391336B50>, 'K_i': <pyomo.core.base.var.ScalarVar object at 0x000001C399CC6050>, 'K_d': <pyomo.core.base.var.ScalarVar object at 0x000001C399CC7350>}]


AttributeError: 'str' object has no attribute 'lb'

In [6]:
# ----------------------- 求场景真值 v(y) -----------------------
def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    del_components(model)
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(pyo.value(v))
    model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)

    try:
        # 若是 persistent，先绑定，再不传 model 求解；否则按 file-based 调用
        if hasattr(solver, "set_instance"):
            solver.set_instance(model)
            results = solver.solve(tee=False)
        else:
            results = solver.solve(model, tee=False)

        status_ok = (results.solver.status == SolverStatus.ok)
        term_ok   = (results.solver.termination_condition == TerminationCondition.optimal)
        if not (status_ok and term_ok):
            raise RuntimeError(
                f"Scenario evaluate at y={first_stg_vals} not optimal: "
                f"status={results.solver.status}, term={results.solver.termination_condition}"
            )
        return pyo.value(model.obj_expr)
    finally:
        if hasattr(model, 'obj'):
            model.del_component('obj')
        for u in first_stg_vars:
            if u.fixed:
                u.unfix()


# ---- 生成“轴向等距插值点”（）----
def build_full_grid_points(first_stg_vars, n_per_axis=10, endpoint=True, round_ndigits=12):
    """
    生成全维等距笛卡尔网格：
    - first_stg_vars: [Kp, Ki, Kd, ...]，每个变量需要有 lb/ub
    - n_per_axis: int 或 序列（逐维个数），例如 10 或 [10,12,8]
    - endpoint: 是否包含右端点（np.linspace 的参数）
    - round_ndigits: 为了稳定，把坐标四舍五入到固定小数位，避免浮点毛刺造成“相同点判不同”
    返回：list[tuple]，长度为 ∏ n_per_axis
    """
    lbs = [float(v.lb) for v in first_stg_vars]
    ubs = [float(v.ub) for v in first_stg_vars]
    d = len(first_stg_vars)

    if isinstance(n_per_axis, int):
        counts = [n_per_axis] * d
    else:
        if len(n_per_axis) != d:
            raise ValueError("n_per_axis 的长度必须等于维度数")
        counts = list(n_per_axis)

    # 每一维的等距坐标
    grids = [
        np.linspace(lbs[i], ubs[i], counts[i], endpoint=endpoint)
        for i in range(d)
    ]

    # 笛卡尔积 -> 点
    pts = [
        tuple(round(float(x), round_ndigits) for x in coords)
        for coords in it.product(*grids)
    ]
    return pts

In [ ]:
# build 3d plot for fsv
csv_path = "data.csv"
max_scenarios = 3
n_per_axis = 5
N = max_scenarios
weights = (1.0, 0.01)
bounds = {
    "x": (-1e3, 1e3),
    "u": (None, None),
    "e": (-1e3, 1e3),
    "I": (-1e5, 1e5),
    "Kp": (0, 100),
    "Ki": (0, 100),
    "Kd": (0, 100),
}
solver = SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 1e-2,
    'FeasibilityTol': 1e-6,
    'IntFeasTol':     1e-2,
    'OptimalityTol':  1e-6,
    'NumericFocus':   0,
    'Presolve':       2,
    'NonConvex':      2,
    'TimeLimit':      120,
})

model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)


# calculate nodes with their values
first_stg_nodes = build_full_grid_points(first_stg_vars_list[0], n_per_axis=n_per_axis, endpoint=True, round_ndigits=12)
node_vals = []
for pt in first_stg_nodes:
    val = 0
    for i in range(N):
        val += evaluate_Q_at(model_list[i], first_stg_vars_list[i], pt, solver)
    node_vals.append(val)


def scatter_colored_points(first_stg_nodes, node_vals, mode="mpl", highlight_min=True):
    """
    mode: 'mpl' 或 'plotly'
    返回: (best_point, best_value)
    """
    pts = np.asarray(first_stg_nodes, dtype=float)
    vals = np.asarray(node_vals, dtype=float).ravel()
    if pts.size == 0 or vals.size == 0:
        raise ValueError("first_stg_nodes / node_vals 不能为空")
    if pts.shape[1] != 3:
        raise ValueError("first_stg_nodes 需为 N×3 的数组")

    # 最小值点
    idx = int(np.argmin(vals))
    best_point, best_value = pts[idx], float(vals[idx])

    if mode == "plotly":
 
        import pandas as pd
        import plotly.express as px
        import plotly.graph_objects as go
        df = pd.DataFrame({"Kp": pts[:,0], "Ki": pts[:,1], "Kd": pts[:,2], "Value": vals})
        fig = px.scatter_3d(df, x="Kp", y="Ki", z="Kd",
                            color="Value", color_continuous_scale="Viridis",
                            opacity=0.85)
        if highlight_min:
            fig.add_trace(go.Scatter3d(
                x=[best_point[0]], y=[best_point[1]], z=[best_point[2]],
                mode='markers',
                marker=dict(size=8, color='red', symbol='diamond'),
                name='Min Point'
            ))
        fig.update_layout(scene=dict(xaxis_title="Kp", yaxis_title="Ki", zaxis_title="Kd"),
                            title=f"Min value = {best_value:.6g} at {best_point}")
        fig.show()


    if mode == "mpl":
        import matplotlib.pyplot as plt
        fig = plt.figure(figsize=(7,6))
        ax = fig.add_subplot(111, projection='3d')
        sc = ax.scatter(pts[:,0], pts[:,1], pts[:,2],
                        c=vals, cmap='viridis', s=40, alpha=0.8)
        if highlight_min:
            ax.scatter(best_point[0], best_point[1], best_point[2],
                       c='red', s=120, marker='*', label='Min Point', depthshade=False)
            ax.legend(loc="best")
        ax.set_xlabel("Kp"); ax.set_ylabel("Ki"); ax.set_zlabel("Kd")
        cb = fig.colorbar(sc, ax=ax, label="Value")
        ax.set_title(f"Min value = {best_value:.6g} at {best_point}")
        plt.tight_layout(); plt.show()

    return best_point, best_value

bp, bv = scatter_colored_points(first_stg_nodes, node_vals, mode="plotly")